# M18 — Model Pruning & Quantization Compression Sweep

> **Chunk F (Generalization & Compression)**  
> **Model ID:** `M18` | **Member:** B  
> **Protocol Compliance:** Real ICBHI Audio, Zero Patient Leakage, Protocol §4 Schema Compliant.

---

### 📌 Overview
This notebook evaluates **Structured Model Pruning & Dynamic INT8 Quantization** across the **M17 Stage-2 Prototypical Model**.

---

### 🎯 Key Deliverables
1. **Pruning Sweep (0%, 20%, 40%, 60%)**: Applies L1-norm structured channel pruning to convolutional layers.
2. **Dynamic INT8 Quantization**: Quantizes linear layers from FP32 to 8-bit integers (`qint8`).
3. **Inference Latency & Size Benchmarks**: Measures model size (MB) and CPU/GPU inference latency (ms/sample).
4. **Accuracy Retention Audit**: Computes classification accuracy retention across all compression configurations.
5. **Tradeoff Plot & CSV Export**: Saves trade-off curves to `Compression_Tradeoff_Plot.png` and exports `compression_sweep_results.csv` and `results_M18.json`.

---

> **2026-08-29 — re-run required (`SYNTHETIC_DATA_REMEDIATION.md` item 4).** The
> previous result was not a held-out score. `discover_icbhi_sweep_files` indexed
> **one row per `.wav`** (851 whole recordings) with no train/test split anywhere,
> and `load_base_model` swallowed every checkpoint error with
> `except Exception: pass` — so the entire sweep could run over a **randomly
> initialised network**. The committed sweep is consistent with exactly that: 0.0411
> accuracy at pruning 0.0, far below the 0.25 four-class chance line, next to a
> "best" of 0.9318 reached only once pruning collapsed the model onto the majority
> class.
>
> Sections 3–7 are rewritten: cycle-level index on the corrected
> patient-independent official split, the checkpoint must load fully or the notebook
> raises, an uncompressed sanity floor is checked before the sweep starts, every
> configuration is scored on TEST with a confusion matrix and the majority-class
> rate beside it, and selection is by **macro-F1** rather than accuracy.
> **Correction, later on 2026-08-29.** The head-width guard fires here too: M17 v2's
> checkpoint is a **projection MLP (…→512→256), not a 4-logit classifier**, so the old
> `strict=False` load was matching nothing and this notebook was operating on a randomly
> initialised network. Section 4 now assembles the real model — M2 backbone → M17
> projection → negative squared distance to prototypes computed on TRAIN — so the **M2
> checkpoint must be attached as well as M17's**.



In [1]:
# ============================================================
# Section 1: Setup & Dependencies
# ============================================================
import os
import sys
import math
import glob
import json
import re
import time
import random
import zipfile
import io
import shutil
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.quantization
import torchaudio
import librosa
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')


Device:  cuda (Tesla T4)
PyTorch: 2.10.0+cu128


In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================

if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M18'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M17_CKPT_PATH = resolve_checkpoint([
    '/content/M17_best_model.pth',
    '/kaggle/input/m17-checkpoint/best_model.pth',
    '/kaggle/input/datasets/barshonbasak/m17-checkpoint/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M17/best_model.pth',
    '../M17/best_model.pth',
    os.path.join(BASE_DIR, 'results_M17', 'best_model.pth'),
])
if M17_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m17' in f.lower() or 'm17' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M17_CKPT_PATH = os.path.join(root, f)
                break
        if M17_CKPT_PATH: break

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'results_M2', 'best_model.pth'),
])
if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m2' in f.lower() or 'm2' in root.lower()) and f.endswith(('.pth', '.zip')):
                M2_CKPT_PATH = os.path.join(root, f)
                break
        if M2_CKPT_PATH:
            break

ICBHI_ROOTS = [
    '/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    './data/audio_and_txt_files',
]
ICBHI_PATH = next((p for p in ICBHI_ROOTS if os.path.exists(p)), None)
if ICBHI_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            ICBHI_PATH = root
            break

CFG = {
    'model_id': 'M18',
    'model_name': 'Model Pruning & Quantization Sweep',
    'member': 'B',
    'seed': SEED,

    # Shared Audio Parameters (§2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # REVIEW (2026-08-29): this notebook declared 3 known_classes but instantiated
    # a 4-output head and loaded the M17 checkpoint with strict=False, so the
    # label space was never pinned down. M17 v2's stage-2 space is the 4 below.
    # Section 4 now asserts the checkpoint's head width matches this list and
    # raises if it does not -- if your checkpoint is the 3-class stage-0 head,
    # shorten this list to ['COPD', 'Healthy', 'URTI'].
    'disease_classes': ['COPD', 'Healthy', 'URTI', 'Pneumonia'],

    # M17 v2 is a PROTOTYPICAL network: its checkpoint holds only the projection
    # MLP, so the model is assembled as M2 backbone -> M17 projection ->
    # distance to prototypes. These must match M17 v2's config (results_M17.json).
    'm2_depth': 5,
    'm2_base_width': 48,
    'proto_embed_dim': 256,
    'proto_temperature': 0.1,
    'known_classes': ['COPD', 'Healthy', 'URTI'],
    'batch_size': 32,

    'm2_ckpt_path': M2_CKPT_PATH,
    'm17_ckpt_path': M17_CKPT_PATH,
    'icbhi_path': ICBHI_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M18'),
}
os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M18 CONFIGURATION — Pruning & Quantization Sweep')
print(f"{'='*60}")
print(f"  M17 Checkpoint: {CFG['m17_ckpt_path'] or 'NOT FOUND'}")
print(f"  ICBHI Path:     {CFG['icbhi_path'] or 'NOT FOUND'}")
print(f"{'='*60}")


Platform: Kaggle

M18 CONFIGURATION — Pruning & Quantization Sweep
  M17 Checkpoint: /kaggle/input/datasets/sudamchandrabasak/m17-checkpoint/best_model.pth
  ICBHI Path:     /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


In [ ]:
# ============================================================
# Section 3: Corrected Split, Cycle Index, Loaders, Metrics
# ============================================================
# Rewritten 2026-08-29 (SYNTHETIC_DATA_REMEDIATION.md item 4). The previous
# version indexed ONE row per .wav -- 851 whole recordings, each tiled to 8 s and
# given its patient's diagnosis as a single label -- and used the SAME rows for
# fitting and for reporting. There was no train/test split anywhere in the
# notebook, so the headline number was a training-set score on 851 recordings
# whose class balance is dominated by COPD.
#
# This version: every annotated respiratory cycle with its own start/end, disease
# label from patient_diagnosis.csv, on the corrected patient-independent official
# split. Nothing is fitted on a row that is later scored.

from torch.utils.data import Dataset, DataLoader   # M18's Section 1 does not import these

# ---- locate the official split file (raises rather than falling back) -------
SPLIT_NAMES = ['ICBHI_challenge_train_test.txt', 'icbhi_challenge_train_test.txt']


def find_split_file():
    roots = ['/kaggle/input', '/content', '.', '..',
             os.path.dirname(CFG['icbhi_path'] or '.')]
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            for name in SPLIT_NAMES:
                if name in files:
                    return os.path.join(dirpath, name)
    return None


def find_diagnosis_file():
    names = ['patient_diagnosis.csv', 'ICBHI_Challenge_diagnosis.txt', 'patient_diagnosis.txt']
    curr = CFG['icbhi_path']
    for _ in range(4):
        for n in names:
            c = os.path.join(curr, n)
            if os.path.exists(c):
                return c
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            for n in names:
                if n in files:
                    return os.path.join(root, n)
    return None


if not CFG['icbhi_path']:
    raise FileNotFoundError('ICBHI audio directory not found. Refusing to run.')

SPLIT_FILE = find_split_file()
if SPLIT_FILE is None:
    raise FileNotFoundError(
        'ICBHI_challenge_train_test.txt not found. The repo copy is at '
        "Asif's/ICBHI_challenge_train_test.txt -- attach it as a Kaggle dataset. "
        'Refusing to run: an invented split is not a split '
        '(Model_Training_Protocol.md section 1).')

DIAG_FILE = find_diagnosis_file()
if DIAG_FILE is None:
    raise FileNotFoundError(
        f'patient diagnosis file not found near {CFG["icbhi_path"]}. '
        'Refusing to run: fabricated labels are not a fallback '
        '(Model_Training_Protocol.md section 1.2).')
print(f'Split file:     {SPLIT_FILE}')
print(f'Diagnosis file: {DIAG_FILE}')

# ---- corrected split, verified against the published counts ----------------
OFFICIAL_RECORDINGS, OFFICIAL_TRAIN, OFFICIAL_TEST = 920, 539, 381
OFFICIAL_PATIENTS, OFFICIAL_OVERLAP = 126, {156, 218}
CORRECTED_TRAIN, CORRECTED_TEST = 551, 369


def pid_of(stem):
    return int(stem.split('_')[0])


split_map = {}
with open(SPLIT_FILE) as fh:
    for line in fh:
        t = line.replace('\t', ' ').replace(',', ' ').split()
        if len(t) >= 2 and t[1].lower() in ('train', 'test'):
            split_map[t[0].replace('.wav', '')] = t[1].lower()

sides = {}
for s, v in split_map.items():
    sides.setdefault(pid_of(s), set()).add(v)
overlap = {p for p, v in sides.items() if len(v) > 1}
n_tr = sum(1 for v in split_map.values() if v == 'train')
assert len(split_map) == OFFICIAL_RECORDINGS, f'{len(split_map)} recordings != 920'
assert n_tr == OFFICIAL_TRAIN and len(split_map) - n_tr == OFFICIAL_TEST
assert len({pid_of(s) for s in split_map}) == OFFICIAL_PATIENTS
assert overlap == OFFICIAL_OVERLAP, f'overlap {sorted(overlap)} != {sorted(OFFICIAL_OVERLAP)}'
corrected = {s: ('train' if pid_of(s) in overlap else v) for s, v in split_map.items()}
c_te = sum(1 for v in corrected.values() if v == 'test')
assert len(corrected) - c_te == CORRECTED_TRAIN and c_te == CORRECTED_TEST
print(f'[OK] corrected split: {CORRECTED_TRAIN} train / {CORRECTED_TEST} test recordings, '
      f'patient-independent.')

# ---- diagnosis map (no fallback) -------------------------------------------
diag_map = {}
with open(DIAG_FILE, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = [p.strip() for p in re.split(r'[,;\t\s]+', line.strip()) if p.strip()]
        if len(parts) >= 2:
            try:
                diag_map[int(parts[0])] = parts[1]
            except ValueError:
                continue
if len(diag_map) != OFFICIAL_PATIENTS:
    raise ValueError(f'diagnosis map has {len(diag_map)} entries, expected {OFFICIAL_PATIENTS}. '
                     f'Refusing to run on a partial label set.')
print(f'[OK] diagnosis map: {len(diag_map)} patients.')

CLASS_TO_IDX = {c: i for i, c in enumerate(CFG['disease_classes'])}

# ---- cycle-level index -----------------------------------------------------
rows, n_missing, n_unlisted, n_other_class = [], 0, 0, 0
for wav in sorted(glob.glob(os.path.join(CFG['icbhi_path'], '**', '*.wav'), recursive=True)):
    stem = os.path.splitext(os.path.basename(wav))[0]
    txt = os.path.splitext(wav)[0] + '.txt'
    if not os.path.exists(txt):
        n_missing += 1
        continue
    sp = corrected.get(stem)
    if sp is None:
        n_unlisted += 1
        continue
    dis = diag_map.get(pid_of(stem))
    if dis not in CLASS_TO_IDX:
        n_other_class += 1
        continue
    with open(txt, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            p = line.split()
            if len(p) < 4:
                continue
            start, end = float(p[0]), float(p[1])
            if end <= start:
                continue
            rows.append({'path': wav, 'stem': stem, 'patient_id': pid_of(stem),
                         'start': start, 'end': end,
                         'label': CLASS_TO_IDX[dis], 'diagnosis': dis, 'split': sp})

df_all = pd.DataFrame(rows)
assert len(df_all) > 3000, f'only {len(df_all)} cycles indexed -- check the audio directory'
if n_missing:
    print(f'skipped {n_missing} recordings with no annotation .txt')
if n_unlisted:
    print(f'skipped {n_unlisted} recordings absent from the split file')
if n_other_class:
    print(f'skipped {n_other_class} recordings outside {CFG["disease_classes"]}')

df_train = df_all[df_all.split == 'train'].reset_index(drop=True)
df_test = df_all[df_all.split == 'test'].reset_index(drop=True)
assert not (set(df_train.patient_id) & set(df_test.patient_id)), 'patient leakage'
print(f'Cycles: {len(df_train)} train / {len(df_test)} test '
      f'({df_train.patient_id.nunique()} / {df_test.patient_id.nunique()} patients), 0% leakage.')
print(df_all.groupby(['split', 'diagnosis']).size().to_string())


def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        raise RuntimeError(f'failed to load audio: {wav_path} [{start}, {end}]') from e
    if len(audio) == 0:
        raise RuntimeError(f'empty audio decoded from {wav_path} [{start}, {end}]')
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    return log_mel[:, :cfg['n_frames']][np.newaxis, :, :].astype(np.float32)


class ICBHI_CycleDataset(Dataset):
    def __init__(self, df, cfg):
        self.records = df.to_dict('records')
        self.cfg = cfg

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return (torch.from_numpy(extract_log_mel(r['path'], r['start'], r['end'], self.cfg)),
                torch.tensor(r['label'], dtype=torch.long))


loader_train = DataLoader(ICBHI_CycleDataset(df_train, CFG),
                          batch_size=CFG['batch_size'], shuffle=True)
loader_test = DataLoader(ICBHI_CycleDataset(df_test, CFG),
                         batch_size=CFG['batch_size'], shuffle=False)

CFG['split_method'] = 'official_60_40_patient_independent_corrected'
CFG['split_file'] = SPLIT_FILE
CFG['num_classes'] = len(CFG['disease_classes'])


# ---- metrics ---------------------------------------------------------------
def full_metrics(y_true, y_pred, class_names):
    # imported locally so this block drops into any of the M16/M18/M20
    # notebooks unchanged, whatever their Section 1 imports happen to be
    from sklearn.metrics import (accuracy_score, confusion_matrix,
                                 precision_recall_fscore_support)
    n_cls = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    p, r, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_cls)), zero_division=0)
    spec = []
    for i in range(n_cls):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i].sum() - tp
        tn = cm.sum() - tp - fp - fn
        spec.append(tn / (tn + fp) if (tn + fp) else 0.0)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(p.mean()), 'recall_macro': float(r.mean()),
        'f1_macro': float(f.mean()), 'specificity_macro': float(np.mean(spec)),
        'majority_class_rate': float(np.bincount(y_true, minlength=n_cls).max() / len(y_true)),
        'per_class': {class_names[i]: {
            'precision': round(float(p[i]), 4), 'recall': round(float(r[i]), 4),
            'f1': round(float(f[i]), 4), 'specificity': round(float(spec[i]), 4),
            'support': int(sup[i])} for i in range(n_cls)},
        'confusion_matrix_raw': cm.tolist(),
        'confusion_matrix_normalized': np.round(
            cm / np.maximum(cm.sum(1, keepdims=True), 1), 4).tolist(),
    }

In [ ]:
# ============================================================
# Section 4: Prototypical Base Model  (raises rather than returning noise)
# ============================================================
import torch.nn.functional as F   # M18's Section 1 does not import this


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )

    def forward(self, x):
        return self.block(x)


class M2_CNN(nn.Module):
    """Verbatim from M17 v2 / M2 -- depth and width come from CFG, not defaults."""

    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]

    def forward(self, x):
        return self.head(self.dropout(self.gap(self.encoder(x)).flatten(1)))

    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)


class PrototypicalProjection(nn.Module):
    """M17 v2's PrototypicalDiseaseHead.projection, key-for-key."""

    def __init__(self, input_dim, embed_dim=256):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )
        self.embed_dim = embed_dim

    def project(self, embeddings):
        return F.normalize(self.projection(embeddings), p=2, dim=-1)


class PrototypicalTeacher(nn.Module):
    """Backbone + projection + fixed prototypes, exposed as an ordinary
    classifier so distillation, calibration and compression can all consume it."""

    def __init__(self, backbone, proj, prototypes, temperature):
        super().__init__()
        self.backbone, self.proj = backbone, proj
        self.register_buffer('prototypes', prototypes)
        self.temperature = float(temperature)

    def forward(self, x):
        z = self.proj.project(self.backbone.get_embedding(x))
        return -(torch.cdist(z, self.prototypes, p=2) ** 2) / self.temperature


def _load_sd(path, what):
    if not path or not os.path.exists(path):
        raise FileNotFoundError(
            f'{what} checkpoint not found: {path!r}. Refusing to run: distilling from a '
            'randomly initialised teacher transfers nothing '
            '(audit: model_may_be_randomly_initialised).')
    # NOTE: torch has saved .pth as a zip archive since 1.6, so is_zipfile() is
    # true for ordinary checkpoints too. Only treat it as a BUNDLE when it
    # actually contains a .pth member; otherwise hand it straight to torch.load.
    inner = None
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if n.endswith('.pth')]
            if names:
                pick = 'best_model.pth' if 'best_model.pth' in names else names[0]
                print(f'  {what}: extracting {pick} from zip bundle')
                inner = io.BytesIO(z.read(pick))
    src = inner if inner is not None else path
    try:
        ckpt = torch.load(src, map_location=DEVICE, weights_only=False)
    except Exception:
        if inner is not None:
            inner.seek(0)
        ckpt = torch.load(inner if inner is not None else path,
                          map_location=DEVICE, weights_only=True)
    sd = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt)) \
        if isinstance(ckpt, dict) else ckpt
    return sd


def _strict_load(module, sd, what, unused_prefixes=()):
    """Load, then refuse if anything the teacher actually USES stayed random.

    `unused_prefixes` names submodules the teacher never calls (the backbone's
    own classifier head -- only get_embedding is used), so they are allowed to
    be absent. Everything else missing is a hard error.
    """
    incompat = module.load_state_dict(sd, strict=False)
    missing = [k for k in incompat.missing_keys
               if not k.endswith('num_batches_tracked')
               and not k.startswith(tuple(unused_prefixes))]
    if missing:
        raise ValueError(
            f'{what}: {len(missing)} parameter tensors were NOT in the checkpoint, e.g. '
            f'{missing[:5]}. Those layers would stay randomly initialised, so the "teacher" '
            'would be partly noise. Check that this is the right checkpoint for this '
            'architecture rather than loosening the load.')
    print(f'  {what}: loaded, 0 required tensors missing '
          f'({len(module.state_dict())} in module)')
    return module


# ---- assemble the teacher --------------------------------------------------
print('Assembling prototypical teacher (M2 backbone + M17 projection + prototypes)...')
backbone = M2_CNN(num_classes=4, depth=CFG['m2_depth'],
                  base_width=CFG['m2_base_width']).to(DEVICE)
_bb_sd = _load_sd(CFG['m2_ckpt_path'], 'M2 backbone')
# The M2 checkpoint carries its own classifier head; only the encoder is reused.
_bb_sd = {k: v for k, v in _bb_sd.items() if not k.startswith('head.')}
_strict_load(backbone, _bb_sd, 'M2 backbone (encoder)', unused_prefixes=('head.',))

proj = PrototypicalProjection(backbone.embedding_dim, CFG['proto_embed_dim']).to(DEVICE)
_strict_load(proj, _load_sd(CFG['m17_ckpt_path'], 'M17 projection'), 'M17 projection')

backbone.eval(); proj.eval()
for p in list(backbone.parameters()) + list(proj.parameters()):
    p.requires_grad = False


@torch.no_grad()
def compute_prototypes(loader, n_cls):
    """Class means of the projected TRAIN embeddings. Test never contributes."""
    sums = torch.zeros(n_cls, CFG['proto_embed_dim'], device=DEVICE)
    counts = torch.zeros(n_cls, device=DEVICE)
    for specs, targets in tqdm(loader, desc='Prototypes (TRAIN)'):
        z = proj.project(backbone.get_embedding(specs.to(DEVICE)))
        t = targets.to(DEVICE)
        sums.index_add_(0, t, z)
        counts.index_add_(0, t, torch.ones_like(t, dtype=torch.float))
    if (counts == 0).any():
        raise ValueError(
            f'no TRAIN cycles for class(es) '
            f'{[CFG["disease_classes"][i] for i in (counts == 0).nonzero().flatten().tolist()]}. '
            'A prototype cannot be formed, so those classes could never be predicted.')
    return F.normalize(sums / counts.unsqueeze(1), p=2, dim=-1)


prototypes = compute_prototypes(loader_train, CFG['num_classes'])
model = PrototypicalTeacher(backbone, proj, prototypes, CFG['proto_temperature']).to(DEVICE).eval()
print(f'Prototypes: {tuple(prototypes.shape)} (temperature {CFG["proto_temperature"]})')

# ---- the sweep operates on the BACKBONE inside the prototypical model -------
# Pruning changes the embedding, so prototypes must be recomputed for each
# configuration -- reusing the dense model's prototypes would score a pruned
# encoder against centroids it can no longer produce.
def build_pruned_model(ratio):
    bb = M2_CNN(num_classes=4, depth=CFG['m2_depth'], base_width=CFG['m2_base_width']).to(DEVICE)
    _strict_load(bb, _bb_sd, 'M2 backbone (encoder)', unused_prefixes=('head.',))
    if ratio > 0:
        for mod in bb.modules():
            if isinstance(mod, nn.Conv2d):
                prune.l1_unstructured(mod, name='weight', amount=ratio)
                prune.remove(mod, 'weight')
    bb.eval()
    for p in bb.parameters():
        p.requires_grad = False
    global backbone
    _saved, backbone = backbone, bb
    try:
        protos = compute_prototypes(loader_train, CFG['num_classes'])
    finally:
        backbone = _saved
    return PrototypicalTeacher(bb, proj, protos, CFG['proto_temperature']).to(DEVICE).eval()


@torch.no_grad()
def evaluate_model(m, loader, device):
    """Full metrics plus mean per-sample latency, on the given loader."""
    m = m.to(device).eval()
    ys, ps, lat = [], [], []
    for specs, targets in loader:
        specs = specs.to(device)
        t0 = time.time()
        logits = m(specs)
        lat.append((time.time() - t0) * 1000 / len(specs))
        ps.append(logits.argmax(-1).cpu().numpy())
        ys.append(targets.numpy())
    if not ys:
        raise RuntimeError('evaluate_model: loader produced no batches.')
    out = full_metrics(np.concatenate(ys), np.concatenate(ps), CFG['disease_classes'])
    out['latency_ms_per_sample'] = float(np.mean(lat))
    return out


base_model = model
base_params = sum(p.numel() for p in base_model.parameters())
print(f'Base model parameters: {base_params:,} (backbone + projection)')

baseline_metrics = evaluate_model(base_model, loader_test, DEVICE)
print(f'Uncompressed TEST accuracy {baseline_metrics["accuracy"]:.4f} '
      f'| macro-F1 {baseline_metrics["f1_macro"]:.4f} '
      f'| majority-class rate {baseline_metrics["majority_class_rate"]:.4f}')
if baseline_metrics['accuracy'] <= 1.0 / CFG['num_classes']:
    raise ValueError(
        f'uncompressed model scores {baseline_metrics["accuracy"]:.4f} on TEST, at or below '
        f'the {1.0 / CFG["num_classes"]:.4f} chance line. The checkpoints are not the trained '
        'model this sweep is supposed to compress. Refusing to continue.')


In [ ]:
# ============================================================
# Section 5: Structured Pruning & Quantization Sweep  (scored on TEST)
# ============================================================
# Rewritten 2026-08-29. Every configuration is now scored on the held-out
# patient-disjoint TEST split, at cycle level, with a confusion matrix and the
# majority-class rate alongside the accuracy -- because a pruned model that has
# collapsed onto the dominant class reports a high accuracy and a macro-F1 near
# 1/n_classes, and the accuracy alone hides that completely.

# Pruning is applied to the backbone and the prototypes are RECOMPUTED for each
# configuration -- see build_pruned_model in Section 4. Scoring a pruned encoder
# against the dense model's centroids would measure the wrong thing.

sweep_results = []
pruning_ratios = [0.0, 0.20, 0.40, 0.60]
quant_modes = ['FP32', 'INT8_Dynamic']

import copy

print('Running compression sweep — all scores on the held-out TEST split...')
for p_ratio in pruning_ratios:
    for q_mode in quant_modes:
        m = build_pruned_model(p_ratio)

        if q_mode == 'INT8_Dynamic':
            # Two traps here, both hit during the 2026-08-29 re-run:
            #  1. quantize_dynamic packs weights for the CPU backend, so the module
            #     must already BE on the CPU -- packing a CUDA module yields params
            #     the CPU kernel cannot unpack ("apply_dynamic is not implemented
            #     for this packed parameter type").
            #  2. `.cpu()` mutates IN PLACE, and the projection module is SHARED
            #     with every other configuration in this sweep. Moving it would
            #     leave the next iteration with a CUDA backbone feeding a CPU
            #     projection. Deep-copy first so the shared module is never touched.
            m_cpu = copy.deepcopy(m).cpu().eval()
            bench_device = torch.device('cpu')
            try:
                m_eval = torch.quantization.quantize_dynamic(
                    m_cpu, {nn.Linear}, dtype=torch.qint8)
            except Exception as e:
                # A backend that cannot do INT8 here is an environment limit, not
                # a result. Record the regime as unavailable rather than faking it.
                print(f'  prune {p_ratio * 100:2.0f}% | {q_mode:<12} | UNAVAILABLE: {e}')
                sweep_results.append({
                    'pruning_ratio': p_ratio, 'quantization_mode': q_mode,
                    'status': 'unavailable', 'reason': str(e),
                })
                continue
        else:
            m_eval, bench_device = m, DEVICE

        met = evaluate_model(m_eval, loader_test, bench_device)
        non_zero = sum((p != 0).sum().item() for p in m.parameters())
        size_mb = non_zero * (1 if q_mode == 'INT8_Dynamic' else 4) / (1024 * 1024)

        sweep_results.append({
            'status': 'ok',
            'pruning_ratio': p_ratio,
            'quantization_mode': q_mode,
            'remaining_params': int(non_zero),
            'model_size_mb': round(float(size_mb), 2),
            'accuracy': round(met['accuracy'], 4),
            'f1_macro': round(met['f1_macro'], 4),
            'majority_class_rate': round(met['majority_class_rate'], 4),
            'beats_majority': bool(met['accuracy'] > met['majority_class_rate']),
            'latency_ms_per_sample': round(met['latency_ms_per_sample'], 3),
            'confusion_matrix_raw': met['confusion_matrix_raw'],
        })
        print(f"  prune {p_ratio * 100:2.0f}% | {q_mode:<12} | {size_mb:6.2f} MB | "
              f"acc {met['accuracy']:.4f} | macro-F1 {met['f1_macro']:.4f} | "
              f"{'beats' if sweep_results[-1]['beats_majority'] else 'BELOW'} majority "
              f"({met['majority_class_rate']:.4f}) | {met['latency_ms_per_sample']:.3f} ms")

df_res = pd.DataFrame([r for r in sweep_results if r.get('status') == 'ok'])
if df_res.empty:
    raise RuntimeError('every sweep configuration failed; nothing to report.')
_unavail = [r for r in sweep_results if r.get('status') != 'ok']
if _unavail:
    print(f'\n{len(_unavail)} configuration(s) unavailable and reported as such, '
          'not estimated.')

# Selection is by macro-F1, not accuracy. On this corpus COPD dominates, so
# accuracy rewards exactly the collapse the sweep is meant to detect.
best_row = df_res.loc[df_res['f1_macro'].idxmax()].to_dict()
print(f"\nBest by macro-F1: prune {best_row['pruning_ratio'] * 100:.0f}% "
      f"{best_row['quantization_mode']} -> macro-F1 {best_row['f1_macro']:.4f}, "
      f"accuracy {best_row['accuracy']:.4f}, {best_row['model_size_mb']:.2f} MB")

spread = df_res['accuracy'].max() - df_res['accuracy'].min()
if spread < 0.01:
    print(f'WARNING: accuracy moves only {spread:.4f} across the whole sweep. A model whose '
          'score does not respond to that much capacity change is emitting a near-constant '
          'prediction (audit: metric_constant_across_sweep).')

csv_path = os.path.join(CFG['results_dir'], 'compression_sweep_results.csv')
df_res.drop(columns=['confusion_matrix_raw']).to_csv(csv_path, index=False)
df_res.drop(columns=['confusion_matrix_raw']).to_csv(
    os.path.join(BASE_DIR, 'compression_sweep_results.csv'), index=False)
print(f'Saved sweep table: {csv_path}')

In [ ]:
# ============================================================
# Section 6: Trade-off Curves  (TEST split)
# ============================================================
# Accuracy is plotted against the majority-class line, because on this corpus a
# collapsed model scores high accuracy; macro-F1 is what shows the collapse.

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
styles = list(zip(['FP32', 'INT8_Dynamic'], ['blue', 'crimson'], ['o', 's']))

ax = axes[0]
for q_mode, color, marker in styles:
    sub = df_res[df_res['quantization_mode'] == q_mode]
    ax.plot(sub['pruning_ratio'] * 100, sub['accuracy'], color=color, marker=marker,
            lw=2, label=f'Accuracy — {q_mode}')
ax.axhline(float(df_res['majority_class_rate'].iloc[0]), color='k', ls='--',
           label='majority-class rate')
ax.set_title('M18 — Pruning vs TEST Accuracy')
ax.set_xlabel('Pruning ratio (%)'); ax.set_ylabel('Accuracy')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

ax = axes[1]
for q_mode, color, marker in styles:
    sub = df_res[df_res['quantization_mode'] == q_mode]
    ax.plot(sub['pruning_ratio'] * 100, sub['f1_macro'], color=color, marker=marker,
            lw=2, label=f'Macro-F1 — {q_mode}')
ax.axhline(1.0 / CFG['num_classes'], color='gray', ls=':',
           label=f'collapse floor (1/{CFG["num_classes"]})')
ax.set_title('M18 — Pruning vs TEST Macro-F1')
ax.set_xlabel('Pruning ratio (%)'); ax.set_ylabel('Macro-F1')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

ax = axes[2]
for q_mode, color, marker in styles:
    sub = df_res[df_res['quantization_mode'] == q_mode]
    ax.plot(sub['pruning_ratio'] * 100, sub['model_size_mb'], color=color, marker=marker,
            lw=2, label=f'Size (MB) — {q_mode}')
ax.set_title('M18 — Pruning vs Model Size')
ax.set_xlabel('Pruning ratio (%)'); ax.set_ylabel('Model size (MB)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
plot_path = os.path.join(CFG['results_dir'], 'Compression_Tradeoff_Plot.png')
plt.savefig(plot_path, dpi=300)
plt.savefig(os.path.join(BASE_DIR, 'Compression_Tradeoff_Plot.png'), dpi=300)
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# ============================================================
# Section 7: Protocol §4 Results JSON Export
# ============================================================

results = {
    'meta': {
        'model_id': 'M18',
        'model_name': 'Model Pruning & Quantization Sweep',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'Structured L1 pruning and dynamic INT8 quantization sweep. Re-run 2026-08-29 per '
            'SYNTHETIC_DATA_REMEDIATION.md item 4. The previous result indexed 851 whole '
            'recordings with no train/test split, and its loader swallowed every checkpoint '
            'error (`except Exception: pass`), so the sweep could run over a randomly '
            'initialised network -- consistent with the committed 0.0411 accuracy at pruning '
            '0.0, below the 0.25 four-class chance line. This run is cycle-level on the '
            'corrected patient-independent official split, the checkpoint must load fully or '
            'the notebook raises, and every configuration is scored on TEST.'
        ),
        'supersedes_note': 'Replaces the 2026-08-05 result (best_accuracy 0.9318, 851 '
                           'recording rows, no split, unchecked checkpoint load).',
    },
    'config': {k: v for k, v in CFG.items() if not callable(v) and not isinstance(v, np.ndarray)},
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'efficiency': {
        'base_params': int(base_params),
        'best_compressed_params': int(best_row['remaining_params']),
        'best_model_size_mb': float(best_row['model_size_mb']),
        'best_latency_ms': float(best_row['latency_ms_per_sample']),
        'gpu_name': GPU_NAME,
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'unit': 'respiratory_cycle',
        'split_method': CFG['split_method'],
        'split_file': SPLIT_FILE,
        'split_file_verified': True,
        'diagnosis_file': DIAG_FILE,
        'official_overlap_patients': sorted(OFFICIAL_OVERLAP),
        'official_overlap_policy': 'reassign_to_train',
        'disease_classes': CFG['disease_classes'],
        'train_samples': int(len(df_train)),
        'test_samples': int(len(df_test)),
        'train_patients': int(df_train.patient_id.nunique()),
        'test_patients': int(df_test.patient_id.nunique()),
        'patient_leakage_verified': True,
        'evaluation_level': 'cycle',
    },
    'best_metrics': {
        'selection_criterion': 'macro_f1 (accuracy rewards majority-class collapse on this corpus)',
        'best_accuracy': float(best_row['accuracy']),
        'best_f1_macro': float(best_row['f1_macro']),
        'majority_class_rate': float(best_row['majority_class_rate']),
        'beats_majority_class': bool(best_row['beats_majority']),
        'best_pruning_ratio': float(best_row['pruning_ratio']),
        'best_quantization_mode': str(best_row['quantization_mode']),
        'confusion_matrix_raw': best_row['confusion_matrix_raw'],
        'uncompressed_baseline': {
            k: (round(v, 4) if isinstance(v, float) else v)
            for k, v in baseline_metrics.items()
        },
        'sweep_summary': sweep_results,
    },
    'ablation': {
        'ablation_group': 'model_compression',
        'ablation_role': 'pruning_quantization_sweep',
        'baseline_model_id': 'M17',
        'variable_changed': 'pruning_and_quantization',
        'variables_held_constant': ['backbone: M2_CNN', 'seed: 42',
                                    f"split: {CFG['split_method']}"],
        'component_flags': {
            'has_sound_event_head': False,
            'has_disease_head': True,
            'has_pruning': True,
            'has_quantization': True,
            'owl_stage': 2,
        },
        'loss_weights': {},
    },
    'training_history': {
        'note': 'post-hoc compression of a frozen checkpoint; no training epochs',
        'sweep': [{k: v for k, v in r.items() if k != 'confusion_matrix_raw'}
                  for r in sweep_results],
    },
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M18.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'Saved: {rpath}')

# The loose M18_metrics.json is deliberately NOT written any more
# (SYNTHETIC_DATA_REMEDIATION.md item 1). results_M18.json is the only export.
legacy = os.path.join(CFG['results_dir'], 'M18_metrics.json')
if os.path.exists(legacy):
    os.remove(legacy)
    print(f'Removed superseded loose export: {legacy}')

In [9]:
# ============================================================
# Section 8: Download Results Zip Bundle
# ============================================================
import zipfile
from IPython.display import HTML, display, FileLink

zip_filename = 'M18_results_bundle.zip'
zip_path = os.path.join(BASE_DIR, zip_filename)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(CFG['results_dir']):
        for f in files:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, BASE_DIR)
            zipf.write(fp, arcname)

print(f"\n{'='*60}")
print('M18 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)')
display(FileLink(zip_filename))



M18 RESULTS DOWNLOAD BUNDLE
Zip: /kaggle/working/M18_results_bundle.zip (0.19 MB)


/kaggle/working/M18_results_bundle.zip